# TP2 — Exercise 3

Run the cells from top to bottom. This notebook includes its prerequisite classes.

### Exercise 3: Logging and timing decorators
The wrappers add messages and timing while preserving the wrapped method's name. A failed call is reported and its exception is raised again. The cleaner below extends the previous class so we can reuse its cleaning behavior.

In [1]:
import pandas as pd
from abc import ABC, abstractmethod
from functools import wraps
from time import perf_counter

#### Setup from earlier exercises

In [2]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def read(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        if self.data is None:
            self.read()
        print(self.data.head(n))

In [3]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.select_dtypes(include="number").columns:
            cleaned[column] = cleaned[column].fillna(cleaned[column].mean())
        return cleaned


class FillMode(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.columns:
            modes = cleaned[column].mode()
            if not modes.empty:
                cleaned[column] = cleaned[column].fillna(modes.iloc[0])
        return cleaned


class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

In [4]:
reader = CSVReader("Data/sample_data (2).csv")
raw_data = reader.read()

#### Exercise 3 implementation

In [5]:
def log_action(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print(f"Starting {function.__name__}")
        try:
            result = function(*args, **kwargs)
        except Exception:
            print(f"Failed {function.__name__}")
            raise
        print(f"Finished {function.__name__}")
        return result
    return wrapper


def log_time(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return function(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            print(f"{function.__name__} took {elapsed:.6f} seconds")
    return wrapper


class LoggedDataCleaner(DataCleaner):
    @log_action
    @log_time
    def clean(self, df):
        return super().clean(df)

In [6]:
cleaner = LoggedDataCleaner(FillMean())
clean_data = cleaner.clean(raw_data)

Starting clean
clean took 0.000510 seconds
Finished clean
